# Transformeri

## 1. Provera GPU-a

In [ ]:
import torch

torch.cuda.is_available()

True

## 2. Instalacija

In [ ]:
!pip -q install "transformers>=4.40" datasets accelerate sentence-transformers

## 3. Podaci

In [ ]:
from google.colab import files

_ = files.upload()

Saving Fake.csv to Fake.csv
Saving True.csv to True.csv


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data --unzip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
100% 41.0M/41.0M [00:00<00:00, 110MB/s]



In [ ]:
import pandas as pd

fake = pd.read_csv("data/Fake.csv")
fake["label"] = 1
true = pd.read_csv("data/True.csv")
true["label"] = 0

df = pd.concat([fake, true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df["label"].value_counts()

,count
label,
1,23481
0,21417


## 4. Ciscenje teksta

In [ ]:
import re


def ocisti(tekst):
    tekst = str(tekst)

    tekst = re.sub(r"^\s*[A-Za-z .,'/-]{0,60}\(Reuters\)\s*-\s*", "", tekst)

    tekst = re.sub(r"\(reuters\)", " ", tekst, flags=re.IGNORECASE)
    tekst = re.sub(r"\breuters\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(featured image via|image via|featured image|getty images|pic\.twitter\.com|screen capture|screenshot via|photo by)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(https?://\S+|www\.\S+|\b\S+\.com\b)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"@\w+", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\s+", " ", tekst)
    tekst = tekst.strip()

    return tekst


df["text_ok"] = df["text"].apply(ocisti)

df["duzina"] = df["text_ok"].str.len()
df = df[df["duzina"] >= 40]
df = df.reset_index(drop=True)

len(df)

44078

## 5. Podela na trening i test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text_ok"], df["label"], test_size=0.25,
    stratify=df["label"], random_state=42)

rezultati = {}

len(X_train), len(X_test)

(33058, 11020)

## 6. Dotreniranje DistilBERT-a

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

MODEL = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)


def tokenizuj(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)


ds_train = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
ds_test = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

ds_train = ds_train.map(tokenizuj, batched=True, remove_columns=["text"])
ds_test = ds_test.map(tokenizuj, batched=True, remove_columns=["text"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/33058 [00:00<?, ? examples/s]

Map:   0%|          | 0/11020 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score


def metrike(eval_pred):
    logiti, labele = eval_pred
    pred = np.argmax(logiti, axis=1)

    e = np.exp(logiti - logiti.max(axis=1, keepdims=True))
    verovatnoce = (e / e.sum(axis=1, keepdims=True))[:, 1]

    preciznost, odziv, f1, _ = precision_recall_fscore_support(
        labele, pred, average="binary", zero_division=0)

    return {"accuracy": accuracy_score(labele, pred),
            "precision": preciznost,
            "recall": odziv,
            "f1": f1,
            "roc_auc": roc_auc_score(labele, verovatnoce)}

In [ ]:
import inspect
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    kljuc = {"processing_class": tokenizer}
else:
    kljuc = {"tokenizer": tokenizer}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrike,
    **kljuc,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.020313,0.009895,0.997187,0.997528,0.997000,0.997264,0.999935
2,0.007171,0.008265,0.997641,0.998233,0.997176,0.997704,0.999942


TrainOutput(global_step=2068, training_loss=0.024275080335094113, metrics={'train_runtime': 483.9412, 'train_samples_per_second': 136.62, 'train_steps_per_second': 4.273, 'total_flos': 4379107264770048.0, 'train_loss': 0.024275080335094113, 'epoch': 2.0})

In [ ]:
ocena = trainer.evaluate()

rezultati["DistilBERT"] = {
    "accuracy": ocena["eval_accuracy"],
    "precision": ocena["eval_precision"],
    "recall": ocena["eval_recall"],
    "f1": ocena["eval_f1"],
    "roc_auc": ocena["eval_roc_auc"],
}

rezultati["DistilBERT"]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.007171,0.008265,2,0.997641,0.998233,0.997176,0.997704,0.999942


{'accuracy': 0.9976406533575317,
 'precision': 0.9982332155477032,
 'recall': 0.9971761383692199,
 'f1': 0.9977043969627406,
 'roc_auc': 0.9999420650819937}

## 7. Embedinzi kao ulaz u klasifikator

In [ ]:
from sentence_transformers import SentenceTransformer

koder = SentenceTransformer("all-MiniLM-L6-v2",
                            device="cuda" if torch.cuda.is_available() else "cpu")

E_train = koder.encode(X_train.tolist(), batch_size=128, show_progress_bar=True)
E_test = koder.encode(X_test.tolist(), batch_size=128, show_progress_bar=True)

E_train.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/259 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

(33058, 384)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

modeli = {"Embedinzi + LogReg": LogisticRegression(max_iter=2000),
          "Embedinzi + LinearSVC": LinearSVC()}

for ime, m in modeli.items():
    m.fit(E_train, y_train)
    pred = m.predict(E_test)

    preciznost, odziv, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="binary", zero_division=0)

    rezultati[ime] = {"accuracy": accuracy_score(y_test, pred),
                      "precision": preciznost,
                      "recall": odziv,
                      "f1": f1}

    if hasattr(m, "predict_proba"):
        rezultati[ime]["roc_auc"] = roc_auc_score(y_test, m.predict_proba(E_test)[:, 1])

    print(ime)
    print(classification_report(y_test, pred, target_names=["true", "fake"], digits=4))

Embedinzi + LogReg
              precision    recall  f1-score   support

        true     0.9280    0.9455    0.9366      5354
        fake     0.9475    0.9306    0.9390      5666

    accuracy                         0.9378     11020
   macro avg     0.9377    0.9381    0.9378     11020
weighted avg     0.9380    0.9378    0.9379     11020

Embedinzi + LinearSVC
              precision    recall  f1-score   support

        true     0.9435    0.9513    0.9474      5354
        fake     0.9536    0.9462    0.9499      5666

    accuracy                         0.9486     11020
   macro avg     0.9485    0.9487    0.9486     11020
weighted avg     0.9487    0.9486    0.9486     11020



## 8. Rezultati

In [ ]:
import json

with open("rezultati_colab.json", "w") as f:
    json.dump(rezultati, f, indent=2, default=float)

files.download("rezultati_colab.json")

pd.DataFrame(rezultati).T.round(4)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,accuracy,precision,recall,f1,roc_auc
DistilBERT,0.9976,0.9982,0.9972,0.9977,0.9999
Embedinzi + LogReg,0.9378,0.9475,0.9306,0.9390,0.9844
Embedinzi + LinearSVC,0.9486,0.9536,0.9462,0.9499,NaN
